In [10]:
import numpy as np
import pandas as pd

In [12]:
data = pd.read_csv(
    "/content/IMDB Dataset.csv",
    engine='python'
)
data

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


In [13]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [14]:
data.head(3)

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive


In [15]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [16]:
data["sentiment"].value_counts()

,count
sentiment,
positive,25000
negative,25000


In [17]:
#convert sentiment columns into numeric
data["sentiment"]=data["sentiment"].map({
    "positive":1,
    "negative":0
})

In [18]:
import re
def remove_url(text):
    pattern=re.compile(r'https?://\S+|www\.\S+')
    return pattern.sub(r'',text)

In [19]:
import re
def remove_tags(text):
    pattern=re.compile('<.*?>')
    return pattern.sub(r'',text)

In [20]:
data["review"]=data["review"].apply(remove_url)

In [21]:
data["review"]=data["review"].apply(remove_tags)

In [22]:
import spacy
nlp=spacy.load("en_core_web_sm")

In [91]:
import spacy

nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

clean_reviews = []

for doc in nlp.pipe(data["review"], batch_size=500):
    tokens = [
        token.lemma_.lower()
        for token in doc
        if not token.is_stop and not token.is_punct and token.is_alpha
    ]

    clean_reviews.append(" ".join(tokens))

data["clean_review"] = clean_reviews

In [92]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1,2))

X = tfidf.fit_transform(data["clean_review"])
y = data["sentiment"]

In [93]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000, C=2)
model.fit(X_train, y_train)

In [94]:
y_pred = model.predict(X_test)

In [95]:
from sklearn.metrics import accuracy_score

print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.9329


In [96]:
from sklearn.metrics import classification_report
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.94      0.92      0.93      4961
           1       0.93      0.94      0.93      5039

    accuracy                           0.93     10000
   macro avg       0.93      0.93      0.93     10000
weighted avg       0.93      0.93      0.93     10000

